##Spark Session

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Transformations").enableHiveSupport().getOrCreate()

#Read Parquet File from hdfs


In [3]:
df = spark.read.parquet("hdfs:///staging_zone/Data_parque.parquet")
df.show()

+-------------------+-------------------+-------------+-----------------+------------------+-----------------+------------------+
|     start_datetime|       end_datetime|trip_distance|  start_longitude|    start_latitude|    end_longitude|      end_latitude|
+-------------------+-------------------+-------------+-----------------+------------------+-----------------+------------------+
|2016-03-01 09:00:00|2016-03-01 09:07:55|          2.5|-73.9767456054687|40.765151977539055|-74.0042648315429| 40.74612808227539|
|2016-03-01 09:00:00|2016-03-01 09:11:06|          2.9|-73.9834823608398|40.767925262451165|-74.0059432983398|  40.7331657409668|
|2016-03-01 09:00:00|2016-03-01 09:31:06|        19.98|-73.7820205688476| 40.64480972290039|-73.9745407104492|  40.6757698059082|
|2016-03-01 09:00:00|2016-03-01 09:00:00|        10.78|-73.8634185791015|40.769813537597656|-73.9696502685546|40.757766723632805|
|2016-03-01 09:00:00|2016-03-01 09:00:00|        30.43|-73.9717407226562| 40.7921829223632

#Data Analysis Before Data Cleaning ^^

In [4]:
df.select("start_longitude" , "start_latitude" , "end_longitude" , "end_latitude").show(1001)  #-> remove zeros

+-----------------+------------------+-----------------+------------------+
|  start_longitude|    start_latitude|    end_longitude|      end_latitude|
+-----------------+------------------+-----------------+------------------+
|-73.9767456054687|40.765151977539055|-74.0042648315429| 40.74612808227539|
|-73.9834823608398|40.767925262451165|-74.0059432983398|  40.7331657409668|
|-73.7820205688476| 40.64480972290039|-73.9745407104492|  40.6757698059082|
|-73.8634185791015|40.769813537597656|-73.9696502685546|40.757766723632805|
|-73.9717407226562| 40.79218292236328|-74.1771697998046| 40.69505310058594|
|-74.0171966552734| 40.70538330078125|-73.9780731201171| 40.75578689575195|
|-73.9945831298828| 40.72784805297852|              0.0|               0.0|
|-73.7887725830078| 40.64775848388672|-73.8292083740234|40.712345123291016|
|-73.9582214355468| 40.76464080810546|-73.9678955078125| 40.76290130615234|
|-73.9857788085937| 40.74119186401367|-73.9463500976562| 40.79787826538086|
|-73.9884262

In [5]:
df.filter("trip_distance <= 0.0").show()  #-> remove zeros

+-------------------+-------------------+-------------+-----------------+-----------------+-----------------+------------------+
|     start_datetime|       end_datetime|trip_distance|  start_longitude|   start_latitude|    end_longitude|      end_latitude|
+-------------------+-------------------+-------------+-----------------+-----------------+-----------------+------------------+
|2016-03-10 16:08:26|2016-03-10 16:08:29|          0.0|-73.9021072387695|40.76406478881836|-73.9020919799804|40.764122009277344|
|2016-03-10 16:11:55|2016-03-10 16:13:17|          0.0| -73.902114868164|40.76422119140625|-73.9020767211913| 40.76423263549805|
+-------------------+-------------------+-------------+-----------------+-----------------+-----------------+------------------+



In [6]:
df.select("start_datetime" , "end_datetime").show(1001)  #-> remove which start == end


+-------------------+-------------------+
|     start_datetime|       end_datetime|
+-------------------+-------------------+
|2016-03-01 09:00:00|2016-03-01 09:07:55|
|2016-03-01 09:00:00|2016-03-01 09:11:06|
|2016-03-01 09:00:00|2016-03-01 09:31:06|
|2016-03-01 09:00:00|2016-03-01 09:00:00|
|2016-03-01 09:00:00|2016-03-01 09:00:00|
|2016-03-01 09:00:00|2016-03-01 09:00:00|
|2016-03-01 09:00:00|2016-03-01 09:00:00|
|2016-03-01 09:00:01|2016-03-01 09:16:04|
|2016-03-01 09:00:01|2016-03-01 09:05:00|
|2016-03-01 09:00:01|2016-03-01 09:24:06|
|2016-03-01 09:00:01|2016-03-01 09:02:03|
|2016-03-01 09:00:02|2016-03-01 09:07:49|
|2016-03-01 09:00:02|2016-03-01 09:03:05|
|2016-03-01 09:00:02|2016-03-01 09:09:32|
|2016-03-01 09:00:02|2016-03-01 09:24:11|
|2016-03-01 09:00:02|2016-03-01 09:08:24|
|2016-03-01 09:00:03|2016-03-01 09:09:23|
|2016-03-01 09:00:03|2016-03-01 09:08:34|
|2016-03-01 09:00:03|2016-03-01 09:32:45|
|2016-03-01 09:00:04|2016-03-01 09:03:30|
|2016-03-01 09:00:04|2016-03-01 09

#Data Cleaning

#Remove records which trip_distance <= 0

In [7]:
df = df.filter("trip_distance > 0.0")
df.filter("trip_distance <= 0.0").show()

+--------------+------------+-------------+---------------+--------------+-------------+------------+
|start_datetime|end_datetime|trip_distance|start_longitude|start_latitude|end_longitude|end_latitude|
+--------------+------------+-------------+---------------+--------------+-------------+------------+
+--------------+------------+-------------+---------------+--------------+-------------+------------+



#Remove Records which start_longitude == 0 OR start_latitude == 0 OR end_longitude == 0 OR end_latitude == 0

In [8]:
df = df.filter("start_longitude != 0.0 AND start_latitude != 0.0 AND end_longitude != 0.0 AND end_latitude != 0.0")

#Remove records which start_datetime <= end_datetime

In [9]:
from pyspark.sql.functions import unix_timestamp
df = df.filter("unix_timestamp(start_datetime) < unix_timestamp(end_datetime)")

#Add Trip_id Column auto-generated

In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id
df = df.withColumn("trip_id", row_number().over(Window.orderBy(monotonically_increasing_id())))

In [11]:
df.show()

2025-10-04 14:26:33,198 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+-------------------+-------------+-----------------+------------------+-----------------+------------------+-------+
|     start_datetime|       end_datetime|trip_distance|  start_longitude|    start_latitude|    end_longitude|      end_latitude|trip_id|
+-------------------+-------------------+-------------+-----------------+------------------+-----------------+------------------+-------+
|2016-03-01 09:00:00|2016-03-01 09:07:55|          2.5|-73.9767456054687|40.765151977539055|-74.0042648315429| 40.74612808227539|      1|
|2016-03-01 09:00:00|2016-03-01 09:11:06|          2.9|-73.9834823608398|40.767925262451165|-74.0059432983398|  40.7331657409668|      2|
|2016-03-01 09:00:00|2016-03-01 09:31:06|        19.98|-73.7820205688476| 40.64480972290039|-73.9745407104492|  40.6757698059082|      3|
|2016-03-01 09:00:01|2016-03-01 09:16:04|          6.2|-73.7887725830078| 40.64775848388672|-73.8292083740234|40.712345123291016|      4|
|2016-03-01 09:00:01|2016-03-01 09

#make trip_id -> 1st Column


In [12]:
df = df.select(["trip_id"] + df.columns[:-1])

In [13]:
df.show()

2025-10-04 14:27:28,666 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+-------------------+-------------------+-------------+-----------------+------------------+-----------------+------------------+
|trip_id|     start_datetime|       end_datetime|trip_distance|  start_longitude|    start_latitude|    end_longitude|      end_latitude|
+-------+-------------------+-------------------+-------------+-----------------+------------------+-----------------+------------------+
|      1|2016-03-01 09:00:00|2016-03-01 09:07:55|          2.5|-73.9767456054687|40.765151977539055|-74.0042648315429| 40.74612808227539|
|      2|2016-03-01 09:00:00|2016-03-01 09:11:06|          2.9|-73.9834823608398|40.767925262451165|-74.0059432983398|  40.7331657409668|
|      3|2016-03-01 09:00:00|2016-03-01 09:31:06|        19.98|-73.7820205688476| 40.64480972290039|-73.9745407104492|  40.6757698059082|
|      4|2016-03-01 09:00:01|2016-03-01 09:16:04|          6.2|-73.7887725830078| 40.64775848388672|-73.8292083740234|40.712345123291016|
|      5|2016-03-01 09:00:01|2016-

#Calcuate start_geo_hash and end_geo_hash by H3 Index


In [14]:
!pip install h3
import h3
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
RESOLUTION = 9  # adjust as needed

def latlon_to_h3(lat, lon, res=RESOLUTION):
    if lat is None or lon is None:
        return None
    # for new h3-py versions
    return h3.latlng_to_cell(lat, lon, res)

h3_udf = udf(latlon_to_h3, StringType())

df_with_h3 = (
    df
    .withColumn("start_geo_hash", h3_udf("start_latitude", "start_longitude"))
    .withColumn("end_geo_hash", h3_udf("end_latitude", "end_longitude"))
)


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/usr/local/bin/python3.8 -m pip install --upgrade pip' command.


In [15]:
df_with_h3.show()

2025-10-04 14:28:44,205 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+-------------------+-------------------+-------------+-----------------+------------------+-----------------+------------------+---------------+---------------+
|trip_id|     start_datetime|       end_datetime|trip_distance|  start_longitude|    start_latitude|    end_longitude|      end_latitude| start_geo_hash|   end_geo_hash|
+-------+-------------------+-------------------+-------------+-----------------+------------------+-----------------+------------------+---------------+---------------+
|      1|2016-03-01 09:00:00|2016-03-01 09:07:55|          2.5|-73.9767456054687|40.765151977539055|-74.0042648315429| 40.74612808227539|892a100d6cfffff|892a1072197ffff|
|      2|2016-03-01 09:00:00|2016-03-01 09:11:06|          2.9|-73.9834823608398|40.767925262451165|-74.0059432983398|  40.7331657409668|892a1008b27ffff|892a1072cdbffff|
|      3|2016-03-01 09:00:00|2016-03-01 09:31:06|        19.98|-73.7820205688476| 40.64480972290039|-73.9745407104492|  40.6757698059082|892a103b1cbff

#Calculate Duration Time

In [16]:
from pyspark.sql.functions import unix_timestamp
df_with_duration = df_with_h3.withColumn("trip_duration_time" , unix_timestamp("end_datetime") - unix_timestamp("start_datetime"))

#Final Tripes data frame


In [17]:
from pyspark.sql.functions import col
Tripes = df_with_duration.select("trip_id" , "start_datetime" , "trip_duration_time" , "start_geo_hash" , "end_geo_hash")
Tripes.show()

2025-10-04 14:30:45,094 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+-------------------+------------------+---------------+---------------+
|trip_id|     start_datetime|trip_duration_time| start_geo_hash|   end_geo_hash|
+-------+-------------------+------------------+---------------+---------------+
|      1|2016-03-01 09:00:00|               475|892a100d6cfffff|892a1072197ffff|
|      2|2016-03-01 09:00:00|               666|892a1008b27ffff|892a1072cdbffff|
|      3|2016-03-01 09:00:00|              1866|892a103b1cbffff|892a1077493ffff|
|      4|2016-03-01 09:00:01|               963|892a103b023ffff|892a100ea8fffff|
|      5|2016-03-01 09:00:01|               299|892a100d44bffff|892a100d68fffff|
|      6|2016-03-01 09:00:01|              1445|892a100d203ffff|892a1008d03ffff|
|      7|2016-03-01 09:00:01|               122|892a100d64bffff|892a10725a3ffff|
|      8|2016-03-01 09:00:02|               467|892a1008817ffff|892a1008d07ffff|
|      9|2016-03-01 09:00:02|               183|892a100899bffff|892a1008807ffff|
|     10|2016-03-01 09:00:02

#Add Tripes TO HIVE ^^


In [19]:

Tripes.write.mode("overwrite").saveAsTable("default.Staging_Rides_Geo")

2025-10-04 14:34:41,865 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
